# Comparação de regulons: motivos contra motivos mais ChIP-seq

Compara, TF a TF, os regulons que o pySCENIC infere em duas condições da mesma réplica:

* **D1**: ranking do cisTarget construído só com motivos;
* **D2**: o mesmo ranking acrescido das linhas de ChIP-seq.

As duas condições partem da mesma matriz de adjacência do GRNBoost2, então a diferença entre os
regulons vem do banco de rankings usado na etapa de enriquecimento.

Os experimentos ficam no dicionário `EXPERIMENTOS`, na Seção 1, e são de dois tipos. Nos de
conjunto (`odom`, com CREB1, FOXA2, HNF1A, HNF4A, ONECUT1 e USF1, e `saintandre`, com FOXQ1, TBX2,
YY1, RREB1 e ELF1), o ChIP-seq de vários TFs centrais entra de uma vez e cada réplica tem um único
CSV de D2. Nos de um TF por vez (`pioneiros`, `canonicos`, `canonicos_preditos`), a pasta de cada
réplica guarda um CSV de D2 para cada TF da lista do experimento. Para incluir um experimento novo
basta acrescentar uma entrada em `EXPERIMENTOS`, com `{replica}` e `{name_tf}` nos caminhos.

O notebook tem duas partes independentes. A parte 1 (Seção 3) compara os CSVs de regulons, uma
comparação por réplica e por TF, e é a parte demorada. A parte 2 (Seção 4) reúne essas comparações
em tabelas por experimento, lendo só o que a parte 1 gravou em disco. Cada parte tem o seu próprio
parâmetro de réplicas, então dá para comparar 30 réplicas e consolidar 5.

Para usar: na Seção 1, escolha os experimentos e as réplicas de cada parte, e execute o notebook
inteiro ou só as células de uma das partes. Com `TFS = None` cada experimento usa a sua própria
lista de TFs; uma lista em `TFS` substitui a de todos os experimentos escolhidos. O D1 de cada
réplica é lido uma vez e comparado com cada CSV de D2. Arquivo que não existir é informado e
pulado, sem interromper o lote.

As saídas de cada experimento ficam organizadas por réplica e por comparação:

```
<pasta de saída do experimento>/
    Replica1/
        MOTIF_vs_MplusATF4/      tabelas e figuras desta comparação
        MOTIF_vs_MplusCEBPB/
        ...
    Replica2/
        ...
    consolidado/                 tabelas da parte 2, com todas as réplicas
```

Nos experimentos de conjunto, que têm um único CSV de D2 por réplica, a pasta da comparação é uma
só: `MOTIF_vs_Mplusodom` e `MOTIF_vs_Mplussaintandre`.

## 1. Parâmetros

Tudo o que muda de uma execução para outra está nesta célula.

In [ ]:
# Listas de TFs dos experimentos de um TF por vez
LISTA_TFS_PIONEIROS_LITERATURA = [
    "ATF4", "CEBPB", "JUN",
    "NFYB", "MAFG", "CEBPA", "FOXA1", "TP53", "FOXA3",
    "ATF6", "ATF7", "NFYC",
]
LISTA_TFS_CANONICOS_LITERATURA = ["GATAD1", "MIXL1", "SMAD3", "TFDP2", "ZBTB25", "GATA4", "GATA2"]

LISTA_TFS_CANONICOS_PREDITOS = [
    "E2F4", "ELF4", "ELK1", "ETS1", "ETV6", "SP1", "E2F1", "E2F2", "ELK4", "GMEB2", "SP2", "TFDP1", "ZBTB14", "ZBTB33",
]

LISTA_TFS_PIONEIROS_PREDITOS = ["HNF4A", "CEBPD","RXRB", "MEF2A","CEBPG","NR5A1", "NFIL3", "PBX2", "NR2F1", "ESRRA", "HLF", "TEAD1", "BCL6", "PPARG"]
# O que analisar
EXPERIMENTOS_A_RODAR = ["pioneiros_preditos"]  # chaves de EXPERIMENTOS, ex.: ["odom", "saintandre"]
TFS = None  # None usa a lista de cada experimento; uma lista própria, ex.: ["ATF4", "JUN"], substitui a de todos

# Réplicas de cada parte, definidas separadamente. Um número usa da réplica 1 até ele; uma lista usa só as indicadas.
REPLICAS_ANALISE = 15         # parte 1: réplicas cujos CSVs brutos de regulons serão comparados
REPLICAS_CONSOLIDACAO = None  # parte 2: réplicas a consolidar; None repete REPLICAS_ANALISE

GERAR_FIGURAS = True  # parte 1: com False, os PNGs de cada comparação não são gerados (útil em lotes grandes)

# Caminhos, com {replica} e {name_tf} preenchidos a cada comparação. Experimento cujo caminho de D2 não tem
# {name_tf} tem um único CSV por réplica e não usa lista de TFs. A chave 'tbl' aponta para a tabela de motivos
# do experimento e só é usada na coluna 'aceito' da parte 2. D1 e D2 têm de ser da mesma réplica, isto é, da
# mesma matriz de adjacência do GRNBoost2.
DATASET1_PATH = "../outs/regs_motifs/reg_Hepg2_n{replica}_motif_th25.csv"  # D1: regulons só com motivos

EXPERIMENTOS = {
    # Conjuntos de TFs centrais adicionados de uma só vez
    "odom": {
        "dataset2": "../outs/regs_odom/reg_Hepg2_adj{replica}_motifs_plus_odom_th25.csv",
        "saida": "./Comparacao_TFS_alterados/MOTIFS_VS_ODOM_att",
        "tfs": None,
    },
    "saintandre": {
        "dataset2": "../outs/regs_saintandre/reg_Hepg2_adj{replica}_motifs_plus_saintandre_th25.csv",
        "saida": "./Comparacao_TFS_alterados/MOTIFS_VS_SAINTANDRE",
        "tfs": None,
    },
     "canonicos_literatura": {
                 "dataset2": "../outs/regs_chip/tf_unico/canonicos_literatura/Replica{replica}/reg_Hepg2_adj{replica}_motifs_plus_{name_tf}_th25.csv",
                 "saida": "./Comparacao_TFS_alterados/tf_unico/MOTIFS_VS_CANONICOS_LITERATURA",
                 "tfs": LISTA_TFS_CANONICOS_LITERATURA,
                 "tbl": "../data/tbls/canonicos_literatura/cntf_{name_tf}.tbl",
    },
    # Um TF por vez: um CSV por TF dentro da pasta de cada réplica
    "pioneiros_literatura": {
        "dataset2": "../outs/regs_chip/tf_unico/ptf/Replica{replica}/reg_Hepg2_adj{replica}_motifs_plus_{name_tf}_th25.csv",
        "saida": "./Comparacao_TFS_alterados/tf_unico/MOTIFS_VS_PIONEIROS_LITERATURA",
        "tfs": LISTA_TFS_PIONEIROS_LITERATURA,
        "tbl": "../data/tbls/pioneiros/ptf_unico/motifs_plus_{name_tf}.tbl",
    },
    "canonicos_preditos": {
            "dataset2": "../outs/regs_chip/tf_unico/canonicos_preditos/Replica{replica}/reg_Hepg2_adj{replica}_motifs_plus_{name_tf}_th25.csv",
            "saida": "./Comparacao_TFS_alterados/tf_unico/MOTIFS_VS_CANONICOS_PREDITOS",
            "tfs": LISTA_TFS_CANONICOS_PREDITOS,
            "tbl": "../data/tbls/canonicos/canonicos_preditos/canonico_{name_tf}.tbl",
    },
    "pioneiros_preditos": {
                "dataset2": "../outs/regs_chip/tf_unico/pioneiros_preditos/Replica{replica}/reg_Hepg2_adj{replica}_motifs_plus_{name_tf}_th25.csv",
                "saida": "./Comparacao_TFS_alterados/tf_unico/MOTIFS_VS_PIONEIROS_PREDITOS",
                "tfs": LISTA_TFS_PIONEIROS_PREDITOS,
                "tbl": "../data/tbls/pioneiros_preditos/pptf_{name_tf}.tbl",
        },

    # Um experimento novo segue o mesmo formato, por exemplo:
    # "pioneiros_preditos": {"dataset2": "...{replica}...{name_tf}...", "saida": "...",
    #                        "tfs": LISTA_TFS_PIONEIROS_PREDITOS, "tbl": "...{name_tf}.tbl"},
}

# O CSV de regulons do pySCENIC tem cabeçalho de mais de uma linha, por isso as colunas são identificadas por
# posição (a primeira é 0); as linhas de cabeçalho que sobram não têm targets e são descartadas na leitura
COL_TF = 0       # nome do TF
COL_TARGETS = 8  # lista de targets no formato [('GENE', peso), ...]

TOP_N_GRAFICO = 59  # parte 1: máximo de TFs exibidos no gráfico de variação (None exibe todos)

SUBPASTA_CONSOLIDACAO = "consolidado"  # parte 2: criada dentro da pasta de saída de cada experimento
DPI = 180                              # parte 2: resolução dos heatmaps

## 2. Bibliotecas e funções da parte 1

In [2]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist

try:
    from matplotlib_venn import venn2
except ImportError:   # sem o matplotlib-venn, só o diagrama de Venn deixa de ser gerado
    venn2 = None

In [3]:
def parse_targets(valor):
    """Converte a string de targets do pySCENIC, "[('GENE', peso), ...]", em lista de tuplas (gene, peso)."""
    if isinstance(valor, list):
        return valor
    texto = "" if pd.isna(valor) else str(valor).strip()
    if not (texto.startswith('[') and texto.endswith(']')):
        return []   # linhas de cabeçalho e linhas sem targets caem aqui
    try:
        return ast.literal_eval(texto)
    except (ValueError, SyntaxError):
        print(f"Não foi possível interpretar os targets: {texto[:100]}...")
        return []


def carregar_regulons(caminho, col_tf, col_targets):
    """Lê o CSV de regulons e devolve uma tabela longa com um par TF/target por linha."""
    df = pd.read_csv(caminho)
    pares = []
    for tf, valor in zip(df.iloc[:, col_tf], df.iloc[:, col_targets]):
        pares.extend((tf, gene) for gene, _ in parse_targets(valor))
    if not pares:
        raise ValueError(f"Nenhum target foi lido de {caminho}. Confira COL_TF e COL_TARGETS.")
    return pd.DataFrame(pares, columns=['TF', 'Target'])


def resumir_por_tf(pares):
    """Uma linha por TF com seus targets únicos (na ordem em que aparecem) e a contagem deles."""
    listas = pares.groupby('TF')['Target'].apply(lambda alvos: alvos.drop_duplicates().tolist())
    resumo = listas.to_frame('targets_lista')
    resumo.insert(0, 'quantidade_targets_unicos', listas.map(len))
    return resumo


def comparar_datasets(resumo_d1, resumo_d2):
    """Compara os targets únicos de cada TF presente em D1 e/ou D2, ordenando pela variação total."""
    linhas = []
    for tf in resumo_d1.index.union(resumo_d2.index):
        t1 = set(resumo_d1['targets_lista'].get(tf, []))
        t2 = set(resumo_d2['targets_lista'].get(tf, []))
        n1, n2 = len(t1), len(t2)
        exclusivos1, exclusivos2 = sorted(t1 - t2), sorted(t2 - t1)
        linhas.append({
            'TF': tf,
            'quantidade_dataset1': n1,
            'quantidade_dataset2': n2,
            'diferenca_quantidade': n2 - n1,
            'diferenca_percentual': (n2 - n1) / n1 * 100 if n1 > 0 else float('inf'),
            'similaridade': len(t1 & t2) / max(n1, n2) * 100,
            'targets_comuns': len(t1 & t2),
            'exclusivos_d1': len(exclusivos1),
            'exclusivos_d2': len(exclusivos2),
            'total_variacao': len(exclusivos1) + len(exclusivos2),
            'targets_exclusivos_dataset1': exclusivos1,
            'targets_exclusivos_dataset2': exclusivos2,
        })
    return (pd.DataFrame(linhas)
            .sort_values(['total_variacao', 'diferenca_quantidade', 'TF'], ascending=[False, False, True])
            .reset_index(drop=True))


def grafico_variacao(comparacao, caminho, top_n=None):
    """Barras com a variação total de targets por TF, coloridas pelo sinal de diferenca_quantidade."""
    dados = comparacao[comparacao['total_variacao'] > 0].sort_values(
        ['total_variacao', 'diferenca_quantidade', 'TF'], ascending=[False, False, True])
    if dados.empty:
        print("Nenhum TF com variação de targets: gráfico de variação não gerado")
        return
    if top_n is not None:
        dados = dados.head(top_n)
    dados = dados.iloc[::-1]  # deixa o TF com maior variação no topo

    cores = np.sign(dados['diferenca_quantidade']).map({1: 'forestgreen', -1: 'firebrick', 0: 'dimgray'})
    plt.figure(figsize=(14, max(6, 0.45 * len(dados))))
    barras = plt.barh(dados['TF'], dados['total_variacao'], color=cores.tolist())

    deslocamento = max(0.3, dados['total_variacao'].max() * 0.01)
    for barra, linha in zip(barras, dados.itertuples()):
        plt.text(barra.get_width() + deslocamento, barra.get_y() + barra.get_height() / 2,
                 f"Δ={linha.diferenca_quantidade:+d} | ExD1={linha.exclusivos_d1} | ExD2={linha.exclusivos_d2}",
                 va='center', fontsize=9)

    plt.xlabel('Variação total de targets (Exclusivos D1 + Exclusivos D2)')
    plt.ylabel('TF')
    plt.title('Variação estrutural dos TFs entre D1 e D2')
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.legend(handles=[
        Line2D([0], [0], color='forestgreen', lw=8, label='Diferença > 0 (aumentou em D2)'),
        Line2D([0], [0], color='firebrick', lw=8, label='Diferença < 0 (diminuiu em D2)'),
        Line2D([0], [0], color='dimgray', lw=8, label='Diferença = 0 (mesma quantidade)'),
    ], loc='lower right')
    plt.tight_layout()
    plt.savefig(caminho, dpi=300, bbox_inches='tight')
    plt.close()


def diagrama_venn(tfs_d1, tfs_d2, caminho):
    """Diagrama de Venn dos TFs com regulon em D1 e em D2."""
    plt.figure(figsize=(8, 8))
    venn = venn2([set(tfs_d1), set(tfs_d2)], set_labels=('D1', 'D2'))
    for regiao, alpha in (('10', 0.6), ('01', 0.6), ('11', 0.7)):
        if venn.get_patch_by_id(regiao) is not None:
            venn.get_patch_by_id(regiao).set_alpha(alpha)
    plt.title('Diagrama de Venn dos TFs presentes em D1 e D2')
    plt.tight_layout()
    plt.savefig(caminho, dpi=300, bbox_inches='tight')
    plt.close()

def comparar_e_salvar(resumo_d1, resumo_d2, pasta, top_n, gerar_figuras):
    """Compara D1 e D2, salva as tabelas (e as figuras, se pedido) em pasta e devolve as contagens para o log."""
    comparacao = comparar_datasets(resumo_d1, resumo_d2)
    em_ambos = comparacao[(comparacao['quantidade_dataset1'] > 0) & (comparacao['quantidade_dataset2'] > 0)]
    alterados = comparacao[comparacao['diferenca_quantidade'] != 0]
    # Mudar o número de targets já derruba a similaridade, então o filtro por ela só faz sentido no caso abaixo
    mesma_qtd = comparacao[(comparacao['diferenca_quantidade'] == 0) & (comparacao['similaridade'] < 100)]

    saidas = {
        'estatisticas_completas_dataset1.csv': resumo_d1.reset_index(),
        'estatisticas_completas_dataset2.csv': resumo_d2.reset_index(),
        'analise_targets_comuns.csv': em_ambos[[
            'TF', 'quantidade_dataset1', 'quantidade_dataset2', 'targets_comuns',
            'targets_exclusivos_dataset1', 'targets_exclusivos_dataset2', 'similaridade',
        ]].rename(columns={'quantidade_dataset1': 'targets_dataset1',
                           'quantidade_dataset2': 'targets_dataset2',
                           'similaridade': 'similaridade_targets'}),
        'tfs_com_quantidade_alterada_targets_unicos.csv': alterados[[
            'TF', 'quantidade_dataset1', 'quantidade_dataset2', 'diferenca_quantidade', 'diferenca_percentual',
            'similaridade', 'targets_comuns', 'exclusivos_d1', 'exclusivos_d2', 'total_variacao',
            'targets_exclusivos_dataset1', 'targets_exclusivos_dataset2',
        ]],
        'tfs_mesma_quantidade_nao_similares.csv': mesma_qtd[[
            'TF', 'quantidade_dataset1', 'similaridade', 'targets_comuns', 'exclusivos_d1', 'exclusivos_d2',
            'total_variacao', 'targets_exclusivos_dataset1', 'targets_exclusivos_dataset2',
        ]].rename(columns={'quantidade_dataset1': 'quantidade'}),
    }
    pasta.mkdir(parents=True, exist_ok=True)
    for nome, tabela in saidas.items():
        tabela.to_csv(pasta / nome, index=False)

    if gerar_figuras:
        grafico_variacao(comparacao, pasta / 'grafico_variacao_total_tfs.png', top_n)
        if venn2 is not None:
            diagrama_venn(resumo_d1.index, resumo_d2.index, pasta / 'venn_tfs_d1_d2.png')
    return len(alterados), len(mesma_qtd)

## 3. Parte 1: comparação dos regulons

Para cada réplica, o CSV de D1 é lido uma vez e comparado com o CSV de D2 de cada TF, e cada
comparação vira uma pasta. Depende das seções 1 e 2, usa `REPLICAS_ANALISE` e é a parte demorada.

As métricas de cada TF, com `n_D1` e `n_D2` sendo o número de targets únicos em cada condição:

* `diferenca_quantidade = n_D2 - n_D1`
* `diferenca_percentual = (n_D2 - n_D1) / n_D1 * 100`, que fica `inf` quando o TF não tem regulon
  em D1
* `similaridade = targets_comuns / max(n_D1, n_D2) * 100`
* `exclusivos_d1` e `exclusivos_d2`: targets presentes em uma só das condições
* `total_variacao = exclusivos_d1 + exclusivos_d2`

Cada pasta de comparação recebe:

* `estatisticas_completas_dataset1.csv` e `estatisticas_completas_dataset2.csv`, com os targets
  únicos de cada TF em D1 e em D2;
* `analise_targets_comuns.csv`, com os TFs presentes nas duas condições, seus targets comuns, os
  exclusivos e a similaridade;
* `tfs_com_quantidade_alterada_targets_unicos.csv`, com os TFs cujo número de targets mudou,
  inclusive os que têm regulon em uma só das condições;
* `tfs_mesma_quantidade_nao_similares.csv`, com os TFs que mantiveram o número de targets mas
  trocaram de genes;
* `grafico_variacao_total_tfs.png` e `venn_tfs_d1_d2.png`, quando `GERAR_FIGURAS = True`. O Venn
  depende do pacote `matplotlib-venn`; sem ele, só essa figura deixa de ser gerada.

In [6]:
replicas = range(1, REPLICAS_ANALISE + 1) if isinstance(REPLICAS_ANALISE, int) else REPLICAS_ANALISE
if GERAR_FIGURAS and venn2 is None:
    print("matplotlib-venn não está instalado (pip install matplotlib-venn): os diagramas de Venn não serão gerados\n")

for experimento in EXPERIMENTOS_A_RODAR:
    config = EXPERIMENTOS[experimento]
    # Com {name_tf} no caminho de D2 há um CSV por TF; sem ele, um único CSV por réplica
    if '{name_tf}' in config['dataset2']:
        rotulos = TFS if TFS is not None else config.get('tfs')
        if not rotulos:
            raise ValueError(f"O experimento '{experimento}' precisa de uma lista de TFs ('tfs' ou TFS).")
    else:
        rotulos = [experimento]

    print(f"===== {experimento} =====")
    salvas = nao_encontrados = 0
    for replica in replicas:
        caminho_d1 = Path(DATASET1_PATH.format(replica=replica))
        if not caminho_d1.exists():
            print(f"Replica{replica}: D1 não encontrado em {caminho_d1}, réplica pulada")
            nao_encontrados += 1
            continue
        resumo_d1 = resumir_por_tf(carregar_regulons(caminho_d1, COL_TF, COL_TARGETS))

        for name_tf in rotulos:
            caminho_d2 = Path(config['dataset2'].format(replica=replica, name_tf=name_tf))
            if not caminho_d2.exists():
                print(f"Replica{replica} | {name_tf}: D2 não encontrado em {caminho_d2}")
                nao_encontrados += 1
                continue
            resumo_d2 = resumir_por_tf(carregar_regulons(caminho_d2, COL_TF, COL_TARGETS))
            pasta = Path(config['saida']) / f"Replica{replica}" / f"MOTIF_vs_Mplus{name_tf}"
            n_alterados, n_mesma_qtd = comparar_e_salvar(resumo_d1, resumo_d2, pasta, TOP_N_GRAFICO, GERAR_FIGURAS)
            salvas += 1
            print(f"Replica{replica} | {name_tf}: {n_alterados} TFs com nº de targets alterado, "
                  f"{n_mesma_qtd} com o mesmo nº e conjuntos diferentes")

    print(f"Concluído: {salvas} comparações salvas em {config['saida']}, {nao_encontrados} arquivos não encontrados\n")

===== pioneiros_preditos =====
Replica1 | HNF4A: 32 TFs com nº de targets alterado, 1 com o mesmo nº e conjuntos diferentes
Replica1 | CEBPD: 25 TFs com nº de targets alterado, 0 com o mesmo nº e conjuntos diferentes
Replica1 | RXRB: 31 TFs com nº de targets alterado, 1 com o mesmo nº e conjuntos diferentes
Replica1 | MEF2A: 26 TFs com nº de targets alterado, 1 com o mesmo nº e conjuntos diferentes
Replica1 | CEBPG: 29 TFs com nº de targets alterado, 0 com o mesmo nº e conjuntos diferentes
Replica1 | NR5A1: 29 TFs com nº de targets alterado, 0 com o mesmo nº e conjuntos diferentes
Replica1 | NFIL3: 28 TFs com nº de targets alterado, 0 com o mesmo nº e conjuntos diferentes
Replica1 | PBX2: 23 TFs com nº de targets alterado, 0 com o mesmo nº e conjuntos diferentes
Replica1 | NR2F1: 26 TFs com nº de targets alterado, 0 com o mesmo nº e conjuntos diferentes
Replica1 | ESRRA: 25 TFs com nº de targets alterado, 0 com o mesmo nº e conjuntos diferentes
Replica1 | HLF: 21 TFs com nº de targets 

## 4. Parte 2: consolidação das réplicas

Reúne as comparações da parte 1 em `<pasta de saída>/consolidado/`, uma consolidação por
experimento. Lê apenas arquivos do disco, então basta executar as seções 1 e 2 e as células desta
seção, sem repetir as comparações. As réplicas vêm de `REPLICAS_CONSOLIDACAO` e podem ser um
subconjunto das que a parte 1 processou. Os arquivos da pasta `consolidado` são sobrescritos a
cada execução.

O que é produzido depende do tipo de experimento.

Nos de um TF por vez, o interesse está no regulon do próprio TF cujo ChIP-seq foi acrescentado:

* `dados_brutos_long.csv`, com uma linha por réplica e TF, trazendo `n_motifs` (targets únicos em
  D1), `var`, `exclu1`, `exclu2`, `churn` (`exclu1 + exclu2`), `tem_reguloma_motif`, `recuperacao`
  (regulon que só existe com o ChIP-seq), `aceito` e `raw_original`;
* `matriz_churn.csv` e `matriz_aceito.csv`, com essas duas colunas no formato TF x réplica.

A coluna `aceito` responde se o track de ChIP-seq foi de fato usado pelo cisTarget. O
identificador do track é lido da última linha da tabela de motivos do experimento (chave `tbl`) e
procurado no CSV de regulons de D2. Sem essa chave, ou quando a tabela não é encontrada, a coluna
fica vazia, para não confundir ChIP-seq rejeitado com verificação que não foi feita.

Nos experimentos de conjunto, em que o mesmo grupo de TFs centrais entra em todas as réplicas, o
interesse está no efeito sobre a rede inteira:

* `numero_genes_alvos_motivos.csv` e `numero_genes_alvos_chip.csv`: matrizes TF x réplica com o
  número de genes-alvo em cada condição, em que célula vazia indica TF sem regulon naquela réplica;
* `numero_regulomas_por_replica.csv`: quantos TFs formam regulon em cada réplica e condição;
* `resumo_all_tfs.csv`: por TF, a frequência de regulons em cada condição e as médias de variação
  total e líquida;
* `heatmap_presenca_binario.png` e `heatmap_variacao_targets.png`.

As frequências e as médias do `resumo_all_tfs.csv` são calculadas sobre as réplicas efetivamente
lidas, de modo que consolidar 5 réplicas não devolve um recorte da consolidação de 30.

### 4.1. Funções comuns às duas consolidações

In [7]:
def ler_csv_opcional(caminho):
    """Lê um CSV gerado pela parte 1; devolve None se ele não existir ou estiver vazio."""
    if not caminho.exists():
        return None
    try:
        return pd.read_csv(caminho)
    except pd.errors.EmptyDataError:
        return None


def linha_do_tf(tabela, tf):
    """Primeira linha da tabela referente ao TF, ou None se a tabela não existir ou não contiver o TF."""
    if tabela is None or 'TF' not in tabela.columns:
        return None
    linhas = tabela[tabela['TF'].astype(str).str.strip() == tf]
    return None if linhas.empty else linhas.iloc[0]


ARQUIVOS_DA_COMPARACAO = ('estatisticas_completas_dataset1.csv', 'estatisticas_completas_dataset2.csv',
                          'tfs_com_quantidade_alterada_targets_unicos.csv',
                          'tfs_mesma_quantidade_nao_similares.csv')


def comparacao_completa(pasta):
    """True se a pasta tem os quatro CSVs da parte 1 e False se não tem nenhum; interrompe se faltar parte deles.

    Arquivo sem nenhuma linha é situação normal, não ausência: significa que nenhum TF caiu naquela categoria.
    """
    faltando = [nome for nome in ARQUIVOS_DA_COMPARACAO if not (pasta / nome).exists()]
    if not faltando:
        return True
    if len(faltando) == len(ARQUIVOS_DA_COMPARACAO):
        return False
    raise FileNotFoundError(f"{pasta} está incompleta, faltam: {', '.join(faltando)}. "
                            "Rode a parte 1 para esta comparação antes de consolidar.")

### 4.2. Experimentos de um TF por vez

In [8]:
def registro_do_proprio_tf(pasta, tf):
    """Devolve (n_motifs, var, exclu1, exclu2, raw_original) para o TF cujo ChIP-seq entrou em D2.

    Procura o TF, nesta ordem, entre os que tiveram o número de targets alterado, entre os que mantiveram
    o número com conjuntos diferentes e, por fim, nas estatísticas de D1 (regulon idêntico nas duas condições).
    """
    linha = linha_do_tf(ler_csv_opcional(pasta / 'tfs_com_quantidade_alterada_targets_unicos.csv'), tf)
    if linha is not None:
        n, var = int(linha['quantidade_dataset1']), int(linha['diferenca_quantidade'])
        e1, e2 = int(linha['exclusivos_d1']), int(linha['exclusivos_d2'])
        return n, var, e1, e2, f"{n}, var = {var}; Exclu1 = {e1}, Exclu2 = {e2}"

    linha = linha_do_tf(ler_csv_opcional(pasta / 'tfs_mesma_quantidade_nao_similares.csv'), tf)
    if linha is not None:
        n, e1, e2 = int(linha['quantidade']), int(linha['exclusivos_d1']), int(linha['exclusivos_d2'])
        return n, 0, e1, e2, f"{n}, var = 0; Exclu1 = {e1}, Exclu2 = {e2}"

    linha = linha_do_tf(ler_csv_opcional(pasta / 'estatisticas_completas_dataset1.csv'), tf)
    if linha is not None:  # regulon presente em D1 e idêntico em D2
        n = int(linha['quantidade_targets_unicos'])
        return n, 0, 0, 0, str(n)

    return 0, 0, 0, 0, 'NÃO TEM REGULOMA'  # sem regulon nas duas condições


def track_do_tbl(caminho):
    """Identificador do track de ChIP-seq acrescentado à tabela de motivos (última linha da coluna #motif_id)."""
    tbl = pd.read_csv(caminho, sep='\t', dtype=str, keep_default_na=False)
    if '#motif_id' not in tbl.columns or tbl.empty:
        return None
    return str(tbl['#motif_id'].iloc[-1]).strip()


def chip_foi_aceito(caminho_reg, track):
    """1 se o identificador do track aparece no CSV de regulons de D2, ou seja, se o cisTarget usou o ChIP-seq."""
    return int(track in caminho_reg.read_text(encoding='utf-8-sig', errors='ignore'))


def consolidar_tf_unico(config, rotulos, replicas, pasta_saida):
    """Tabela longa com uma linha por réplica e TF, mais as matrizes TF x réplica de churn e aceito."""
    tracks = {}  # um track por TF, lido uma única vez e reaproveitado em todas as réplicas
    if 'tbl' in config:
        for tf in rotulos:
            caminho_tbl = Path(config['tbl'].format(name_tf=tf))
            tracks[tf] = track_do_tbl(caminho_tbl) if caminho_tbl.exists() else None
            if tracks[tf] is None:
                print(f"  {tf}: track não lido em {caminho_tbl}, coluna 'aceito' fica vazia")
    else:
        print("  experimento sem a chave 'tbl': coluna 'aceito' fica vazia")

    registros = []
    for replica in replicas:
        for tf in rotulos:
            pasta = Path(config['saida']) / f"Replica{replica}" / f"MOTIF_vs_Mplus{tf}"
            if not comparacao_completa(pasta):
                print(f"  Replica{replica} | {tf}: comparação não encontrada em {pasta}")
                continue
            n_motifs, var, e1, e2, raw = registro_do_proprio_tf(pasta, tf)

            # Verificação independente das tabelas acima: o track precisa aparecer no CSV de regulons de D2
            caminho_reg = Path(config['dataset2'].format(replica=replica, name_tf=tf))
            pode_verificar = tracks.get(tf) is not None and caminho_reg.exists()
            registros.append({
                'TF': tf,
                'replica': replica,
                'aceito': chip_foi_aceito(caminho_reg, tracks[tf]) if pode_verificar else pd.NA,
                'n_motifs': n_motifs,
                'var': var,
                'exclu1': e1,
                'exclu2': e2,
                'churn': e1 + e2,
                'tem_reguloma_motif': int(n_motifs > 0),
                'recuperacao': int(n_motifs == 0 and var > 0),  # regulon que só existe com o ChIP-seq
                'raw_original': raw,
            })

    longo = pd.DataFrame(registros)
    if longo.empty:
        print("  nenhuma comparação encontrada")
        return
    longo['aceito'] = longo['aceito'].astype('Int64')  # aceita valores ausentes sem virar float

    pasta_saida.mkdir(parents=True, exist_ok=True)
    longo.to_csv(pasta_saida / 'dados_brutos_long.csv', index=False)
    for coluna, nome in (('churn', 'matriz_churn.csv'), ('aceito', 'matriz_aceito.csv')):
        matriz = longo.pivot(index='TF', columns='replica', values=coluna).reindex(rotulos)
        matriz.columns = [f"REPLICA {c}" for c in matriz.columns]
        matriz.reset_index().to_csv(pasta_saida / nome, index=False)

    nao_verificados = int(longo['aceito'].isna().sum())
    sem_aceite_com_mudanca = int(((longo['aceito'] == 0) & (longo['churn'] > 0)).sum())
    print(f"  {len(longo)} linhas | 'aceito' não verificado: {nao_verificados} | "
          f"ChIP-seq não aceito mas com mudança na rede: {sem_aceite_com_mudanca}")

In [9]:
def frequencia_regulomas(config, rotulos, replicas, pasta_saida):
    """Matrizes TF x réplica de presença do regulon do próprio TF em D1 (motivos) e em D2 (motivos mais ChIP-seq).

    Em cada réplica, o TF recebe 1 se aparece em estatisticas_completas_dataset1.csv (ou dataset2) da pasta da sua
    própria comparação e 0 se não aparece. freq_media é o número de 1 dividido pelo número de réplicas. Réplica sem
    a comparação do TF fica vazia e não entra no denominador.
    """
    registros = []
    for replica in replicas:
        for tf in rotulos:
            pasta = Path(config['saida']) / f"Replica{replica}" / f"MOTIF_vs_Mplus{tf}"
            if not comparacao_completa(pasta):
                continue
            registros.append({
                'TF': tf,
                'replica': f"Replica{replica}",
                'motif': int(linha_do_tf(ler_csv_opcional(pasta / 'estatisticas_completas_dataset1.csv'), tf) is not None),
                'chip': int(linha_do_tf(ler_csv_opcional(pasta / 'estatisticas_completas_dataset2.csv'), tf) is not None),
            })
    if not registros:
        print("  nenhuma comparação encontrada: frequências de regulons não geradas")
        return

    presenca = pd.DataFrame(registros)
    colunas = [f"Replica{replica}" for replica in replicas]
    pasta_saida.mkdir(parents=True, exist_ok=True)
    for condicao, nome in (('motif', 'frequencia_regulomas_motif.csv'), ('chip', 'frequencia_regulomas_chip.csv')):
        matriz = (presenca.pivot(index='TF', columns='replica', values=condicao)
                  .reindex(index=rotulos, columns=colunas))
        matriz['freq_media'] = matriz[colunas].mean(axis=1)  # média só sobre as réplicas com comparação
        for coluna in colunas:
            matriz[coluna] = matriz[coluna].astype('Int64')
        (matriz.rename_axis(index='TF', columns=None).reset_index()
         .to_csv(pasta_saida / nome, index=False, float_format='%.3f'))
    print("  frequência de regulons salva em frequencia_regulomas_motif.csv e frequencia_regulomas_chip.csv")

### 4.3. Experimentos de conjunto

In [10]:
def leituras_por_replica(config, rotulo, replicas):
    """Número de alvos por TF e condição e variação total por TF, lidos das colunas já calculadas na parte 1."""
    contagens, variacoes = [], []
    for replica in replicas:
        nome_replica = f"Replica{replica}"
        pasta = Path(config['saida']) / nome_replica / f"MOTIF_vs_Mplus{rotulo}"
        if not comparacao_completa(pasta):
            print(f"  Replica{replica}: comparação não encontrada em {pasta}")
            continue
        for condicao, nome in (('motivos', 'estatisticas_completas_dataset1.csv'),
                               ('chip', 'estatisticas_completas_dataset2.csv')):
            tabela = pd.read_csv(pasta / nome)
            contagens.append(pd.DataFrame({
                'replica': nome_replica,
                'TF': tabela['TF'].astype(str).str.strip(),
                'condicao': condicao,
                'n_alvos': tabela['quantidade_targets_unicos'],
            }))
        # Um TF aparece em no máximo um dos dois arquivos; fora deles, o regulon é idêntico nas duas condições
        for nome in ('tfs_com_quantidade_alterada_targets_unicos.csv', 'tfs_mesma_quantidade_nao_similares.csv'):
            tabela = ler_csv_opcional(pasta / nome)
            if tabela is not None and not tabela.empty:
                variacoes.append(tabela[['TF', 'total_variacao']].assign(replica=nome_replica))

    if not contagens:
        return pd.DataFrame(), pd.DataFrame()
    variacoes = (pd.concat(variacoes, ignore_index=True).drop_duplicates(['replica', 'TF']) if variacoes
                 else pd.DataFrame(columns=['TF', 'total_variacao', 'replica']))
    return pd.concat(contagens, ignore_index=True), variacoes


def ordenar_e_dimensionar(matriz):
    """Reordena linhas e colunas por agrupamento hierárquico e calcula o tamanho da figura."""
    def ordem(dados):
        if dados.shape[0] < 2:
            return list(range(dados.shape[0]))
        return hierarchy.dendrogram(hierarchy.linkage(pdist(dados), method='ward'), no_plot=True)['leaves']

    dados = matriz.fillna(0).to_numpy()
    matriz = matriz.iloc[ordem(dados), ordem(dados.T)]
    n_tf, n_rep = matriz.shape
    lado = max(0.2, min(0.6, 200 / max(n_tf, n_rep)))
    return matriz, (max(8, n_rep * lado + 4), max(6, n_tf * lado + 2))


def heatmap_presenca(comparacao, replicas_lidas, caminho, dpi):
    """Ganho e perda de regulons por réplica: filtrado pelo ChIP-seq, presente em ambos ou resgatado."""
    presenca = (comparacao.pivot(index='TF', columns='replica', values='presenca')
                .reindex(columns=replicas_lidas).fillna(0))
    matriz, tamanho = ordenar_e_dimensionar(presenca)

    fig, ax = plt.subplots(figsize=tamanho)
    imagem = ax.imshow(matriz.to_numpy(), aspect='auto', interpolation='nearest',
                       cmap=ListedColormap(['#E53935', '#F0F0F0', '#1565C0', '#43A047']),
                       norm=BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5], 4))
    ax.set_xticks(range(matriz.shape[1]))
    ax.set_xticklabels(matriz.columns, rotation=90, fontsize=6)
    ax.set_yticks(range(matriz.shape[0]))
    ax.set_yticklabels(matriz.index, fontsize=6)
    ax.set_xlabel('Réplica', fontsize=11)
    ax.set_ylabel('Fator de Transcrição', fontsize=11)
    ax.set_title('Mudança de presença/ausência dos TFs por réplica', fontsize=12, pad=12)
    barra = fig.colorbar(imagem, ax=ax, ticks=[-1, 0, 1, 2], fraction=0.03, pad=0.02)
    barra.ax.set_yticklabels(['Filtrado pelo ChIP (\u22121)', 'Ausente em ambos (0)',
                              'Presente em ambos (+1)', 'Resgatado pelo ChIP (+2)'], fontsize=8)
    plt.tight_layout()
    plt.savefig(caminho, dpi=dpi, bbox_inches='tight')
    plt.close()


def heatmap_variacao(comparacao, replicas_lidas, caminho, dpi):
    """Variação líquida do número de alvos por TF e réplica."""
    variacao = (comparacao.pivot(index='TF', columns='replica', values='variacao_liquida')
                .reindex(columns=replicas_lidas))
    matriz, tamanho = ordenar_e_dimensionar(variacao)
    # Escala simétrica limitada ao percentil 95 do módulo, para que valores extremos não apaguem o restante
    limite = max(np.nanpercentile(np.abs(matriz.to_numpy()), 95), 1)

    fig, ax = plt.subplots(figsize=tamanho)
    sns.heatmap(matriz, ax=ax, cmap='RdBu_r', center=0, vmin=-limite, vmax=limite,
                linewidths=0.01, linecolor='#eee',
                cbar_kws={'label': 'Δ n_targets (ChIP − Motivos)', 'shrink': 0.6})
    ax.set_xlabel('Réplica', fontsize=11)
    ax.set_ylabel('Fator de Transcrição', fontsize=11)
    ax.set_title('Variação quantitativa no número de alvos por TF', fontsize=12, pad=12)
    ax.tick_params(axis='x', labelsize=6, rotation=90)
    ax.tick_params(axis='y', labelsize=6)
    plt.tight_layout()
    plt.savefig(caminho, dpi=dpi, bbox_inches='tight')
    plt.close()


def consolidar_conjunto(config, rotulo, replicas, pasta_saida, dpi):
    """Reúne as réplicas de um experimento de conjunto e descreve o efeito do ChIP-seq sobre a rede."""
    contagens, variacoes = leituras_por_replica(config, rotulo, replicas)
    if contagens.empty:
        print("  nenhuma réplica encontrada")
        return
    replicas_lidas = list(dict.fromkeys(contagens['replica']))
    n_replicas = len(replicas_lidas)

    # Uma linha por réplica e TF, com o número de alvos em cada condição; vazio indica TF sem regulon
    comparacao = (contagens.pivot_table(index=['replica', 'TF'], columns='condicao', values='n_alvos')
                  .reindex(columns=['motivos', 'chip']).rename_axis(columns=None).reset_index())
    tem_motivos, tem_chip = comparacao['motivos'].notna(), comparacao['chip'].notna()
    comparacao['presenca'] = np.select([tem_motivos & tem_chip, tem_motivos], [1, -1], default=2)
    comparacao['variacao_liquida'] = comparacao['chip'].fillna(0) - comparacao['motivos'].fillna(0)
    comparacao = comparacao.merge(variacoes, on=['replica', 'TF'], how='left')
    comparacao['variacao_total'] = comparacao.pop('total_variacao').fillna(0)

    pasta_saida.mkdir(parents=True, exist_ok=True)
    # Matrizes TF x réplica com o número de genes-alvo; célula vazia indica TF sem regulon na réplica
    for condicao, nome in (('motivos', 'numero_genes_alvos_motivos.csv'), ('chip', 'numero_genes_alvos_chip.csv')):
        (contagens[contagens['condicao'] == condicao]
         .pivot(index='TF', columns='replica', values='n_alvos')
         .reindex(columns=replicas_lidas).astype('Int64')
         .reset_index().to_csv(pasta_saida / nome, index=False))

    (contagens.groupby(['replica', 'condicao']).size().unstack(fill_value=0)
     .reindex(index=replicas_lidas, columns=['motivos', 'chip'], fill_value=0)
     .rename(columns={'motivos': 'n_reg_controle', 'chip': 'n_reg_tratado'})
     .rename_axis(index='Replicas', columns=None).reset_index()
     .to_csv(pasta_saida / 'numero_regulomas_por_replica.csv', index=False))

    # Fração do total de réplicas em que cada TF forma regulon, por condição
    frequencia = (contagens.groupby(['TF', 'condicao']).size().unstack(fill_value=0)
                  .reindex(columns=['motivos', 'chip'], fill_value=0) / n_replicas)
    medias = comparacao.groupby('TF')[['variacao_total', 'variacao_liquida']].mean()
    resumo = pd.DataFrame({
        'freq_motif': frequencia['motivos'],
        'freq_chip': frequencia['chip'],
        'variacao_total_media': medias['variacao_total'],
        'variacao_liquida_media': medias['variacao_liquida'],
    }).rename_axis('TF').reset_index()
    resumo.to_csv(pasta_saida / 'resumo_all_tfs.csv', index=False, float_format='%.3f')

    heatmap_presenca(comparacao, replicas_lidas, pasta_saida / 'heatmap_presenca_binario.png', dpi)
    heatmap_variacao(comparacao, replicas_lidas, pasta_saida / 'heatmap_variacao_targets.png', dpi)
    print(f"  {n_replicas} réplicas | {len(resumo)} TFs | "
          f"{int((resumo['variacao_total_media'] > 0).sum())} com variação total média maior que zero")

### 4.4. Execução

In [11]:
# Réplicas desta parte, independentes das comparações da parte 1
escolha = REPLICAS_ANALISE if REPLICAS_CONSOLIDACAO is None else REPLICAS_CONSOLIDACAO
replicas_consolidacao = range(1, escolha + 1) if isinstance(escolha, int) else escolha

for experimento in EXPERIMENTOS_A_RODAR:
    config = EXPERIMENTOS[experimento]
    pasta_saida = Path(config['saida']) / SUBPASTA_CONSOLIDACAO
    print(f"===== consolidação: {experimento} =====")

    if '{name_tf}' in config['dataset2']:
        rotulos = TFS if TFS is not None else config.get('tfs')
        if not rotulos:
            raise ValueError(f"O experimento '{experimento}' precisa de uma lista de TFs ('tfs' ou TFS).")
        consolidar_tf_unico(config, rotulos, replicas_consolidacao, pasta_saida)
        frequencia_regulomas(config, rotulos, replicas_consolidacao, pasta_saida)
    else:
        consolidar_conjunto(config, experimento, replicas_consolidacao, pasta_saida, DPI)

    print(f"Consolidação salva em {pasta_saida}\n")

===== consolidação: pioneiros_preditos =====
  210 linhas | 'aceito' não verificado: 0 | ChIP-seq não aceito mas com mudança na rede: 0
  frequência de regulons salva em frequencia_regulomas_motif.csv e frequencia_regulomas_chip.csv
Consolidação salva em Comparacao_TFS_alterados/tf_unico/MOTIFS_VS_PIONEIROS_PREDITOS/consolidado

